# Lab 3（下）換成真的股票，同一把尺再量一次

上半場那 100 家公司是**我們自己編的**——親手放了一條規則進去，所以切開來考，模型還有 0.80～0.87。

**真實市場沒有人放規則進去。** 這半場換上 8 檔真的台灣股票，**同樣四種模型、同樣切一刀**，看看數字會變成什麼樣。

> 上半場學會的動作原封不動再做一次，**只有資料換了**。

In [ ]:
# 📦 先跑這一格：指定版本，避免學校電腦裝到不相容的舊版（裝不起來看 README）
!pip install -q scikit-learn==1.6.1 pandas==2.2.3 numpy==1.26.4 matplotlib==3.10.0 xgboost==3.2.0 yfinance==1.3.0

## 🔧 第 0 步：環境檢查

⚠️ 下半場**需要連網**——後半段要上網抓真實股價。這一格會順便試抓一次，**有問題現在就會知道，不會等到課程後半才爆**。

In [ ]:
# 老師的小設定：載入今天整本要用的工具、設好中文字型（直接跑、不用改）
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

plt.rcParams["font.sans-serif"] = ["Noto Sans CJK JP", "Noto Sans CJK TC", "Microsoft JhengHei", "PingFang TC", "AR PL UMing CN", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False

import sklearn
import xgboost
import yfinance as yf                      # 抓股價用，後半段會用到

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

print("sklearn   ", sklearn.__version__)
print("pandas    ", pd.__version__)
print("matplotlib", matplotlib.__version__)
print("xgboost   ", xgboost.__version__)
print("yfinance  ", yf.__version__)

### 順便試抓一次，確認連得上

抓台積電最近幾天就好。**印得出價格 → 後半段沒問題；印不出來 → 看最下面的 Backup。**

In [ ]:
test = yf.download("2330.TW", period="5d", auto_adjust=True, progress=False)

if len(test) > 0:
    print("✅ 連得上，抓到", len(test), "天")
    print("最新一天：", test.index[-1].date())
else:
    print("❌ 抓不到資料 —— 檢查網路，或過幾分鐘再試（看最下面的 Backup）")

---
## 🟩 H・抓 8 檔真實台股

### H1・上網抓資料、算技術指標

`yfinance` 第 0 步已經載入過了，這次一次抓 8 檔、算 5 個技術指標。**每次跑都上網抓到今天、不存快取 → 數字每次不同，看結構別背數字。**

In [ ]:
tickers = ["2330.TW", "2317.TW", "2454.TW", "2308.TW",
           "2382.TW", "2412.TW", "2881.TW", "2882.TW"]
name_map = {"2330.TW": "台積電", "2317.TW": "鴻海", "2454.TW": "聯發科",
            "2308.TW": "台達電", "2382.TW": "廣達", "2412.TW": "中華電",
            "2881.TW": "富邦金", "2882.TW": "國泰金"}

raw = yf.download(tickers, start="2022-01-01", auto_adjust=True, progress=False)

parts = []
for code_ in tickers:
    one = pd.DataFrame({"close": raw["Close"][code_], "vol": raw["Volume"][code_]}).dropna()
    one["ret5"] = one["close"].pct_change(5)                    # 近 5 日漲跌幅
    one["ret20"] = one["close"].pct_change(20)                  # 近 20 日漲跌幅
    ma20 = one["close"].rolling(20).mean()
    one["ma_bias"] = one["close"] / ma20 - 1                    # 現價離均線多遠
    one["vol20"] = one["close"].pct_change().rolling(20).std()  # 近 20 日波動大小
    one["volr"] = one["vol"] / one["vol"].rolling(20).mean()    # 成交量比平常放大幾倍
    one["label"] = (one["close"].shift(-1) > one["close"]).astype(int)   # 明天漲＝1
    one = one.iloc[:-1]     # 最後一天沒有「明天」可對答案 → 去掉
    one = one.dropna()
    one["ticker"] = name_map[code_]
    one["date"] = one.index
    parts.append(one[["date", "ticker", "ret5", "ret20", "ma_bias", "vol20", "volr", "label"]])

data = pd.concat(parts).sort_values("date").reset_index(drop=True)

feature_names = ["ret5", "ret20", "ma_bias", "vol20", "volr"]
print("總筆數：", len(data))
print("資料最新日期：", data["date"].max().date())
data.head()

### H2・⚠️ 這次不能隨機切

股價有**時間順序**。隨機打散＝可能拿 3 月的資料去考 1 月的題目，等於**偷看未來**。

**正確做法：找一個分界日，之前的念書、之後的當考卷。**

In [ ]:
data = data.sort_values("date").reset_index(drop=True)

# TODO：填一個小數，代表「前面幾成拿去訓練」—— 本課用八成
cut = int(len(data) * ____)
split_date = data["date"].iloc[cut]    # 用這一天當分界

# TODO：填兩個比較符號 —— 分界日「之前」的念書、分界日「當天以後」的當考卷
#       ⚠️ 兩份加起來要剛好是全部：不能重疊，也不能漏掉分界日那一天
train = data[data["date"] ____ split_date]
test = data[data["date"] ____ split_date]

X_train2 = train[feature_names].values
y_train2 = train["label"].values
X_test2 = test[feature_names].values
y_test2 = test["label"].values

print("分界日期：", split_date.date())
print("樣本內（較早的日子）：", len(X_train2), "筆")
print("樣本外（較晚的日子）：", len(X_test2), "筆")

### H3・⭐ 同樣四種，再考一次

In [ ]:
models2 = [
    ("KNN k=1      ", KNeighborsClassifier(n_neighbors=1)),
    ("KNN k=5      ", KNeighborsClassifier(n_neighbors=5)),
    ("KNN k=15     ", KNeighborsClassifier(n_neighbors=15)),
    ("決策樹 深度2  ", DecisionTreeClassifier(max_depth=2, random_state=42)),
    ("決策樹 不限深 ", DecisionTreeClassifier(max_depth=None, random_state=42)),
    ("隨機森林      ", RandomForestClassifier(n_estimators=200, random_state=42)),
    ("XGBoost      ", XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.3,
                                    random_state=42, eval_metric="logloss")),
]

for name, model in models2:
    model.fit(X_train2, y_train2)
    in_score = model.score(X_train2, y_train2)
    out_score = model.score(X_test2, y_test2)
    print(name, "樣本內", round(in_score, 3), " 樣本外", round(out_score, 3),
          " 差距", round(in_score - out_score, 3))

**🔑 ⭐⭐ 把這張表跟上半場 F3 那張擺在一起——同一群模型、同一個動作，結論整個翻過來：**

- **樣本外全部塌在 0.50 附近**，跟擲銅板一樣。編的資料上還有 0.80～0.87。
- **隨機森林從全場最高變成差距最大之一**——資料本身沒訊號時，它照背不誤，**不是萬靈丹**。
- **XGBoost 是唯一沒把樣本內衝到 1.000 的**，因為一開始就給它 `max_depth=3`、深度鎖住了。而它的樣本外還是最高的。

**➡️ 這不是做錯了，是這題真的很難**——光靠技術指標猜隔日漲跌，本來就接近擲銅板。

---
## 🟧 I・把旋鈕從一端轉到另一端

### I1・掃 KNN 的 k

In [ ]:
ks = [1, 3, 5, 7, 9, 15, 25, 49]

print("k | 樣本內 | 樣本外 | 差距")
for k in ks:
    # TODO：KNN 的 k 在 sklearn 裡不是寫成 k —— 參數名回頭看上半場建 KNN 那格
    m = KNeighborsClassifier(____=k)
    m.fit(X_train2, y_train2)
    in_score = m.score(X_train2, y_train2)
    out_score = m.score(X_test2, y_test2)
    gap = round(in_score - out_score, 3)
    if gap > 0.1:
        flag = "  ← 過擬合!"
    else:
        flag = ""
    print(k, "|", round(in_score, 3), "|", round(out_score, 3), "|", gap, flag)

### I2・換一種模型、換一個旋鈕

In [ ]:
depths = [1, 2, 3, 5, 10, None]     # None ＝ 不限制深度

print("max_depth | 樣本內 | 樣本外 | 差距")
for d in depths:
    # TODO：決策樹「最多問幾層」的參數名 —— 回頭看上半場建決策樹那格
    m = DecisionTreeClassifier(____=d, random_state=42)
    m.fit(X_train2, y_train2)
    in_score = m.score(X_train2, y_train2)
    out_score = m.score(X_test2, y_test2)
    gap = round(in_score - out_score, 3)
    if gap > 0.1:
        flag = "  ← 過擬合!"
    else:
        flag = ""
    print(d, "|", round(in_score, 3), "|", round(out_score, 3), "|", gap, flag)

**🔑 ⭐⭐ 兩張表放一起看：`k=1` 和 `不限深` 那兩行，樣本內都是 1.000、樣本外都崩。**

**兩種完全不同的模型、完全不同的旋鈕，卻用同一種方式壞掉——所以過擬合不是某種模型的毛病，是通則。**

碰到任何模型、任何旋鈕，都先問這一句：**樣本外呢？**

> ⚠️ **但不是每個旋鈕都會「轉過頭」。** 森林的棵數就不會——**越多越穩，只是跑越慢**，設個夠大的數字就好，不用調。

### I3・四種模型，各挑一組能用的設定

**三條規則：**
1. 樣本內幾乎全對的（> 0.99）→ **背死了，不能選**
2. 樣本內外差距 > 0.1 的 → **已經在過擬合，不能選**
3. 剩下的裡面，挑**樣本外最高**的

先把候選設定列出來。

In [ ]:
candidates = []

for k in ks:
    candidates.append(("KNN", "k=" + str(k), KNeighborsClassifier(n_neighbors=k)))

for d in depths:
    candidates.append(("決策樹", "深度" + str(d),
                       DecisionTreeClassifier(max_depth=d, random_state=42)))

for d in [2, 3, 5, 10, None]:
    candidates.append(("隨機森林", "深度" + str(d),
                       RandomForestClassifier(n_estimators=200, max_depth=d, random_state=42)))

for d in [1, 2, 3, 5]:
    candidates.append(("XGBoost", "深度" + str(d),
                       XGBClassifier(n_estimators=200, max_depth=d, learning_rate=0.3,
                                     random_state=42, eval_metric="logloss")))

print("一共", len(candidates), "組設定要試")

**⚠️ 森林和 XGBoost 這裡調的是「深度」不是「棵數」**——棵數越多越穩、不會轉過頭，深度才是會壞的那個旋鈕。

In [ ]:
best = {}      # 每一種模型記一組最好的

for family, tag, m in candidates:
    m.fit(X_train2, y_train2)
    in_score = m.score(X_train2, y_train2)
    out_score = m.score(X_test2, y_test2)

    if in_score > 0.99:                 # 規則1：背死的
        continue
    if in_score - out_score > 0.1:      # 規則2：差距太大＝過擬合
        continue
    if family not in best:              # 規則3：這一種還沒有紀錄，先記下來
        best[family] = (tag, out_score, m)
    # TODO：什麼情況才該換掉舊紀錄？—— 新這組的樣本外要比舊的「怎樣」
    elif out_score ____ best[family][1]:   #  已經有紀錄，比它好才換掉
        best[family] = (tag, out_score, m)

for family in best:
    tag = best[family][0]
    out_score = best[family][1]
    print(family, "→", tag, "  樣本外", round(out_score, 3))

**🔑 四種模型各挑出一組能用的設定，樣本外全部只比擲銅板好一點點。** 用正確的方法挑，這題還是很難——**誠實的結論就是這樣。**

**🔑 這一段是同一個迴圈跑完四種模型**——挑設定的方法從頭到尾只有一套，換模型不換方法。

### I4・哪個技術指標比較有用

In [ ]:
d3 = DecisionTreeClassifier(max_depth=3, random_state=42)
d3.fit(X_train2, y_train2)

for i in range(len(feature_names)):
    # TODO：決策樹的招牌屬性 —— 每個欄位有多重要。⚠️ 結尾有一個底線
    print(feature_names[i], "重要性：", round(d3.____[i], 3))

**🔑 五個指標把重要性分掉了，沒有單一指標獨大**——真實資料長這樣。**以後每多一個候選線索，都能回來看它排第幾。**

### 📝 小作業：換一個切分比例

把切分比例從 0.8 改成 0.7，重挑一次 KNN 的 k。**挑到的 k 會不會變？**

In [ ]:
# 📝 小作業參考解（參考解不只一種，思路對就好）

cut70 = int(len(data) * 0.7)
sd70 = data["date"].iloc[cut70]
tr70 = data[data["date"] < sd70]
te70 = data[data["date"] >= sd70]

Xtr70 = tr70[feature_names].values
ytr70 = tr70["label"].values
Xte70 = te70[feature_names].values
yte70 = te70["label"].values

best_k70 = None
best_out70 = -1
for k in ks:
    m = KNeighborsClassifier(n_neighbors=k)
    m.fit(Xtr70, ytr70)
    in_score = m.score(Xtr70, ytr70)
    out_score = m.score(Xte70, yte70)
    if in_score > 0.99:
        continue
    if in_score - out_score > 0.1:
        continue
    if out_score > best_out70:
        best_out70 = out_score
        best_k70 = k

print("切 80% 時挑到的 KNN：", best["KNN"][0])
print("切 70% 時挑到的 KNN： k=" + str(best_k70))

**🔑 換個切法，挑到的 k 就變了。** 所以**能帶走的是方法，不是「k 要設幾」這個數字。**

---
## 🎬 J・收尾

### 誠實衡量，是一條貫穿整門課的線

| 何時 | 那把尺 |
|---|---|
| 課程 1 | precision / recall —— 撈得準不準、有沒有漏 |
| 上半場 | 樣本內 vs 樣本外，在自己編的資料上第一次切開考 |
| 這半場 | 同一把尺量真實股票 —— **四種模型、掃旋鈕、挑設定** |
| **下一個模組・GA** | **過度最佳化**——在歷史上調到完美 ≠ 未來會賺 |

**模型一直換，尺從不換。**

### 分類模型在金融上的定位

**是輔助參考，最終決策要靠人。** 今天真正學到的本事是「**會建模型 ＋ 會誠實驗證模型**」。

> 🎓 **證照接點：** KNN、決策樹、隨機森林、特徵重要性、過擬合／欠擬合、train/test 分割都是常考題，尤其「決策樹好解釋 vs 神經網路黑盒」。
>
> 🔧 業界也有 **Weka** 這種不用寫程式、點選就能做資料探勘的工具。本課統一用 Python，知道有這種東西存在即可。

---
## 🥚 彩蛋：真的來預測「明天」

前面所有的樣本外，答案其實**早就在資料裡**了。這一格不一樣——**用最新一天的指標猜下一個交易日，答案明天才會揭曉。**

**四種模型全部上場**，各自用剛剛挑出來的那組設定。

In [ ]:
# 先把每一檔「最新一天」的技術指標算出來
today_rows = []
for code_ in tickers:
    o = pd.DataFrame({"close": raw["Close"][code_], "vol": raw["Volume"][code_]}).dropna()
    o["ret5"] = o["close"].pct_change(5)
    o["ret20"] = o["close"].pct_change(20)
    ma20 = o["close"].rolling(20).mean()
    o["ma_bias"] = o["close"] / ma20 - 1
    o["vol20"] = o["close"].pct_change().rolling(20).std()
    o["volr"] = o["vol"] / o["vol"].rolling(20).mean()
    o = o.dropna()
    last = o.iloc[-1]                       # 最新那一天（有指標、但沒有明天）
    today_rows.append({
        "股票": name_map[code_], "資料日": o.index[-1].date(),
        "ret5": last["ret5"], "ret20": last["ret20"], "ma_bias": last["ma_bias"],
        "vol20": last["vol20"], "volr": last["volr"],
    })
today = pd.DataFrame(today_rows)

print("最新資料日：", today["資料日"][0])
today[["股票", "ret5", "ret20", "ma_bias"]].round(4)

In [ ]:
# 玩真的：用全部有答案的資料重新訓練（不留考卷），四種各猜一次
X_all = data[feature_names].values
y_all = data["label"].values
X_today = today[feature_names].values

families = []
for family in best:
    tag = best[family][0]
    m = best[family][2]
    m.fit(X_all, y_all)                     # 用全部資料重新訓練
    today[family] = m.predict(X_today)
    families.append(family)
    print(family, "用的設定：", tag)

In [ ]:
def updown(v):
    if v == 1:
        return "漲"
    else:
        return "跌"

print("📅 用", today["資料日"][0], "收盤後的指標，猜每檔『下一個交易日』漲跌：")
print()
for i in range(len(today)):
    line = today["股票"][i] + "  "
    for family in families:
        line = line + family + ":" + updown(today[family][i]) + "  "
    print(line)
print()
print("⏰ 明天收盤後回來對照真實漲跌，看誰猜對了")
print("（樣本外只有 5 成 → 這是一場誠實的擲銅板）")

**🔑 看四種會不會給一樣的答案。** 不一致的那幾檔最有意思——**同一份資料、同一個問題，四種模型看法不同**，這時候誰也不能說自己是對的。

**⚠️ 這裡是「玩真的」：用全部資料訓練、不留考卷。** 但也因此**沒有任何分數可以拿來相信它**——樣本外告訴我們的是 5 成，這一格只是把它演出來。

---
## 🛟 Backup：卡住的時候

| 狀況 | 怎麼辦 |
|---|---|
| `No module named 'xgboost'` | `pip install xgboost==3.2.0`；裝不動改用 Colab |
| 抓不到 Yahoo Finance | 過幾分鐘再試（很多人同時抓可能被擋）。**這半場整段都需要網路** |
| 「我的數字跟老師不一樣」 | **本來就會不一樣**——H 段每次抓到今天為止的最新資料。看結構別背數字 |
| 技術指標是負的 | 正常，跌就是負的 |
| `max_depth=None` 怎麼寫 | `None` 不加引號，不是字串 `"None"` |